# TasteMap — Colab backup compute

Backup path for Phase 2 (base embeddings) if local compute is too slow.

Usage:
1. Zip `data/images/` and `data/tastemap.db` and upload to Drive (or upload directly below).
2. Run all cells. Output `.npy` + id list get written to Drive; copy them back into `data/cache/` locally.

In [ ]:
!pip install -q open_clip_torch pillow tqdm numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever you uploaded the images folder in Drive.
IMAGES_DIR = '/content/drive/MyDrive/tastemap/images'
OUT_DIR = '/content/drive/MyDrive/tastemap/cache'

import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import torch, open_clip, glob
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# Keep in sync with config.CLIP_MODEL_NAME / config.CLIP_PRETRAINED
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'laion2b_s34b_b79k'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device).eval()
print('device:', device)

In [ ]:
paths = sorted(glob.glob(os.path.join(IMAGES_DIR, '*')))
ids = [os.path.splitext(os.path.basename(p))[0] for p in paths]
print(f'{len(paths)} images found')

In [ ]:
BATCH = 32
all_embs = []

with torch.no_grad():
    for i in tqdm(range(0, len(paths), BATCH)):
        batch_paths = paths[i:i+BATCH]
        imgs = torch.stack([preprocess(Image.open(p).convert('RGB')) for p in batch_paths]).to(device)
        feats = model.encode_image(imgs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embs.append(feats.cpu().numpy())

embeddings = np.concatenate(all_embs, axis=0)
print(embeddings.shape)

In [ ]:
np.save(os.path.join(OUT_DIR, 'embeddings.npy'), embeddings)
with open(os.path.join(OUT_DIR, 'ids.txt'), 'w') as f:
    f.write('\n'.join(ids))
print('saved to', OUT_DIR)